# 电影情感分析

1.导入库

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torch.utils.tensorboard import SummaryWriter
import os
import re
import time
from datetime import datetime
from collections import Counter
import urllib.request
import tarfile
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string

# 下载NLTK资源
try:
    nltk.download('stopwords')
except:
    pass

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


[nltk_data] Downloading package stopwords to /home/magia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2.数据准备

In [8]:
# 设置超参数
BATCH_SIZE = 64
EMBEDDING_DIM = 300
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.5
LEARNING_RATE = 0.001
NUM_EPOCHS = 15  # 增加训练轮数
MAX_VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 300
MIN_FREQ = 5

# 初始化TensorBoard
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter(f'runs/imdb_sentiment_analysis_{timestamp}')

# 手动下载和处理IMDB数据集
def download_and_extract_imdb():
    url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    filename = "aclImdb_v1.tar.gz"
    data_dir = "aclImdb"
    
    if not os.path.exists(data_dir):
        print("Downloading IMDB dataset...")
        urllib.request.urlretrieve(url, filename)
        
        print("Extracting IMDB dataset...")
        with tarfile.open(filename, 'r:gz') as tar:
            tar.extractall()
        
        os.remove(filename)
        print("IMDB dataset downloaded and extracted successfully!")
    
    return data_dir

# 加载IMDB数据集
def load_imdb_data(data_dir):
    reviews = []
    labels = []
    
    # 加载正面评价
    pos_dir = os.path.join(data_dir, 'train', 'pos')
    for filename in os.listdir(pos_dir):
        with open(os.path.join(pos_dir, filename), 'r', encoding='utf-8') as f:
            reviews.append(f.read())
            labels.append(1)  # 正面评价标签为1
    
    # 加载负面评价
    neg_dir = os.path.join(data_dir, 'train', 'neg')
    for filename in os.listdir(neg_dir):
        with open(os.path.join(neg_dir, filename), 'r', encoding='utf-8') as f:
            reviews.append(f.read())
            labels.append(0)  # 负面评价标签为0
    
    return reviews, labels

# 改进的文本预处理函数
def preprocess_text(text):
    # 转换为小写
    text = text.lower()
    
    # 移除HTML标签
    text = re.sub(r'<[^>]+>', '', text)
    
    # 移除URL
    text = re.sub(r'http\S+', '', text)
    
    # 处理缩写
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'t", " not", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'m", " am", text)
    
    # 保留字母和基本标点
    text = re.sub(r'[^a-zA-Z\s!?\']', ' ', text)
    
    # 移除多余空格
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# 创建词汇表
def build_vocab(texts, max_vocab_size, min_freq=5):
    counter = Counter()
    for text in texts:
        tokens = text.split()
        counter.update(tokens)
    
    # 过滤低频词
    counter = {word: count for word, count in counter.items() if count >= min_freq}
    
    # 获取最常见的词汇
    most_common = sorted(counter.items(), key=lambda x: x[1], reverse=True)[:max_vocab_size-2]
    
    # 创建词汇表
    vocab = {'<pad>': 0, '<unk>': 1}
    for idx, (word, _) in enumerate(most_common, start=2):
        vocab[word] = idx
    
    # 添加情感关键词（如果不在词汇表中）
    sentiment_words = ['excellent', 'great', 'amazing', 'wonderful', 'fantastic', 
                      'terrible', 'awful', 'horrible', 'bad', 'disappointing']
    
    for word in sentiment_words:
        if word not in vocab:
            vocab[word] = len(vocab)
    
    return vocab

# 文本编码函数
def text_to_sequence(text, vocab, max_length):
    tokens = text.split()
    sequence = [vocab.get(token, vocab['<unk>']) for token in tokens]
    
    # 截断或填充序列
    if len(sequence) > max_length:
        sequence = sequence[:max_length]
    else:
        sequence = sequence + [vocab['<pad>']] * (max_length - len(sequence))
    
    return sequence

# 自定义数据集类
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        sequence = text_to_sequence(text, self.vocab, self.max_length)
        return torch.tensor(sequence, dtype=torch.long), torch.tensor(label, dtype=torch.long)

# 下载和处理IMDB数据集
data_dir = download_and_extract_imdb()
reviews, labels = load_imdb_data(data_dir)

# 检查数据平衡性
print(f"Positive reviews: {sum(labels)}")
print(f"Negative reviews: {len(labels) - sum(labels)}")

# 预处理文本
print("Preprocessing texts...")
preprocessed_reviews = [preprocess_text(review) for review in reviews]

# 构建词汇表
print("Building vocabulary...")
vocab = build_vocab(preprocessed_reviews, MAX_VOCAB_SIZE, MIN_FREQ)
print(f"Vocabulary size: {len(vocab)}")

# 检查一些情感关键词是否在词汇表中
sentiment_words = ['excellent', 'great', 'amazing', 'wonderful', 'fantastic', 
                  'terrible', 'awful', 'horrible', 'bad', 'disappointing']
for word in sentiment_words:
    if word in vocab:
        print(f"'{word}' in vocabulary at index {vocab[word]}")
    else:
        print(f"'{word}' NOT in vocabulary")

# 划分训练集和测试集
train_texts, test_texts, train_labels, test_labels = train_test_split(
    preprocessed_reviews, labels, test_size=0.2, random_state=42, stratify=labels
)

# 进一步划分训练集和验证集
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42, stratify=train_labels
)

# 创建数据集
train_dataset = IMDBDataset(train_texts, train_labels, vocab, MAX_SEQUENCE_LENGTH)
val_dataset = IMDBDataset(val_texts, val_labels, vocab, MAX_SEQUENCE_LENGTH)
test_dataset = IMDBDataset(test_texts, test_labels, vocab, MAX_SEQUENCE_LENGTH)

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Positive reviews: 12500
Negative reviews: 12500
Preprocessing texts...
Building vocabulary...
Vocabulary size: 20000
'excellent' in vocabulary at index 309
'great' in vocabulary at index 87
'amazing' in vocabulary at index 472
'wonderful' in vocabulary at index 375
'fantastic' in vocabulary at index 774
'terrible' in vocabulary at index 383
'awful' in vocabulary at index 361
'horrible' in vocabulary at index 513
'bad' in vocabulary at index 81
'disappointing' in vocabulary at index 1301


3.网络模型

In [9]:
# 改进的RNN模型
class ImprovedRNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, model_type='lstm', bidirectional=True):
        super(ImprovedRNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.model_type = model_type
        self.bidirectional = bidirectional
        
        if model_type == 'lstm':
            self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers, 
                              batch_first=True, dropout=dropout, bidirectional=bidirectional)
        elif model_type == 'gru':
            self.rnn = nn.GRU(embed_dim, hidden_dim, num_layers,
                             batch_first=True, dropout=dropout, bidirectional=bidirectional)
        
        self.dropout = nn.Dropout(dropout)
        
        # 计算全连接层的输入维度
        fc_input_dim = hidden_dim * 2 if bidirectional else hidden_dim
        
        # 更深的分类器
        self.classifier = nn.Sequential(
            nn.Linear(fc_input_dim, fc_input_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_input_dim // 2, fc_input_dim // 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_input_dim // 4, 2)
        )
        
        # 初始化权重
        self._init_weights()
    
    def _init_weights(self):
        # 初始化嵌入层权重
        nn.init.xavier_uniform_(self.embedding.weight)
        # 对<pad>标记的嵌入向量初始化为0
        self.embedding.weight.data[0] = torch.zeros(self.embedding.embedding_dim)
        
        # 初始化RNN权重
        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
        
        # 初始化分类器权重
        for layer in self.classifier:
            if hasattr(layer, 'weight'):
                nn.init.xavier_uniform_(layer.weight)
    
    def forward(self, x):
        embedded = self.embedding(x)
        
        if self.model_type == 'lstm':
            output, (hidden, _) = self.rnn(embedded)
        elif self.model_type == 'gru':
            output, hidden = self.rnn(embedded)
        
        # 处理双向RNN的输出
        if self.bidirectional:
            # 对于双向RNN，取最后时间步的前向和后向隐藏状态
            if self.model_type == 'lstm':
                hidden_forward = hidden[-2, :, :]
                hidden_backward = hidden[-1, :, :]
                hidden = torch.cat((hidden_forward, hidden_backward), dim=1)
            else:
                hidden_forward = hidden[-2, :, :]
                hidden_backward = hidden[-1, :, :]
                hidden = torch.cat((hidden_forward, hidden_backward), dim=1)
        else:
            # 对于单向RNN，取最后一个隐藏状态
            hidden = hidden[-1, :, :]
        
        out = self.dropout(hidden)
        return self.classifier(out)

# 学习率调度器
def get_lr_scheduler(optimizer, scheduler_type='plateau'):
    if scheduler_type == 'step':
        return optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
    elif scheduler_type == 'plateau':
        return optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    elif scheduler_type == 'cosine':
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=0)
    else:
        return None

4.训练

In [10]:
# 训练函数
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, model_name, scheduler=None):
    best_acc = 0.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # 训练阶段
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            # 每100个batch记录一次
            if batch_idx % 100 == 0:
                writer.add_scalar(f'Loss/batch_train_{model_name}', loss.item(), epoch * len(train_loader) + batch_idx)
        
        train_accuracy = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)
        
        # 验证阶段
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_accuracy = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        
        # 更新学习率
        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(avg_val_loss)
            else:
                scheduler.step()
        
        # 记录到TensorBoard
        writer.add_scalar(f'Loss/train_{model_name}', avg_train_loss, epoch)
        writer.add_scalar(f'Loss/val_{model_name}', avg_val_loss, epoch)
        writer.add_scalar(f'Accuracy/train_{model_name}', train_accuracy, epoch)
        writer.add_scalar(f'Accuracy/val_{model_name}', val_accuracy, epoch)
        
        # 记录学习率
        current_lr = optimizer.param_groups[0]['lr']
        writer.add_scalar(f'LearningRate/{model_name}', current_lr, epoch)
        
        # 保存最佳模型
        if val_accuracy > best_acc:
            best_acc = val_accuracy
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f'best_model_{model_name}.pth')
        
        # 记录时间
        epoch_time = time.time() - start_time
        
        print(f'Epoch [{epoch+1}/{num_epochs}], {model_name}')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%')
        print(f'Learning Rate: {current_lr:.6f}')
        print(f'Time: {epoch_time:.2f}s')
        print('-' * 60)
    
    # 加载最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model

# 评估函数
def evaluate_model(model, test_loader, model_name):
    model.eval()
    test_correct = 0
    test_total = 0
    all_labels = []
    all_predictions = []
    all_probabilities = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    test_accuracy = 100 * test_correct / test_total
    print(f'{model_name} Test Accuracy: {test_accuracy:.2f}%')
    
    # 记录到TensorBoard
    writer.add_scalar(f'Accuracy/test_{model_name}', test_accuracy)
    
    # 计算并绘制混淆矩阵
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Negative', 'Positive'], 
                yticklabels=['Negative', 'Positive'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix - {model_name}')
    writer.add_figure(f'Confusion_Matrix/{model_name}', plt.gcf())
    
    # 打印分类报告
    print(f"Classification Report for {model_name}:")
    print(classification_report(all_labels, all_predictions, 
                               target_names=['Negative', 'Positive']))
    
    return test_accuracy, all_labels, all_predictions, all_probabilities

5.评估

In [11]:
# 训练和评估LSTM模型
print("Training LSTM model...")
lstm_model = ImprovedRNNModel(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, 'lstm', bidirectional=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = get_lr_scheduler(optimizer, 'plateau')

lstm_model = train_model(lstm_model, train_loader, val_loader, criterion, optimizer, NUM_EPOCHS, 'lstm', scheduler)
lstm_accuracy, lstm_labels, lstm_predictions, lstm_probabilities = evaluate_model(lstm_model, test_loader, 'lstm')

# 训练和评估GRU模型
print("Training GRU model...")
gru_model = ImprovedRNNModel(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, 'gru', bidirectional=True).to(device)
optimizer = optim.Adam(gru_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = get_lr_scheduler(optimizer, 'plateau')

gru_model = train_model(gru_model, train_loader, val_loader, criterion, optimizer, NUM_EPOCHS, 'gru', scheduler)
gru_accuracy, gru_labels, gru_predictions, gru_probabilities = evaluate_model(gru_model, test_loader, 'gru')

# 添加模型结构到TensorBoard
dummy_input = torch.randint(0, len(vocab), (1, MAX_SEQUENCE_LENGTH)).to(device)
writer.add_graph(lstm_model, dummy_input)
writer.add_graph(gru_model, dummy_input)

# 添加嵌入可视化
# 注意：由于词汇表可能很大，我们只可视化前1000个词
subset_vocab = {k: v for k, v in list(vocab.items())[:1000]}
subset_metadata = list(subset_vocab.keys())
subset_indices = list(subset_vocab.values())

# 获取嵌入权重
embedding_weights = lstm_model.embedding.weight.data.cpu()
subset_embeddings = embedding_weights[subset_indices]

writer.add_embedding(subset_embeddings, metadata=subset_metadata, tag="LSTM Embeddings")

# 同样为GRU模型添加嵌入可视化
embedding_weights = gru_model.embedding.weight.data.cpu()
subset_embeddings = embedding_weights[subset_indices]
writer.add_embedding(subset_embeddings, metadata=subset_metadata, tag="GRU Embeddings")

# 关闭TensorBoard写入器
writer.close()

# 情感预测函数
def predict_sentiment(model, text, vocab, max_length):
    model.eval()
    with torch.no_grad():
        # 预处理文本
        processed_text = preprocess_text(text)
        # 转换为序列
        sequence = text_to_sequence(processed_text, vocab, max_length)
        # 转换为张量
        input_tensor = torch.tensor(sequence, dtype=torch.long).unsqueeze(0).to(device)
        # 预测
        output = model(input_tensor)
        probabilities = torch.softmax(output, dim=1)
        _, predicted = torch.max(output, 1)
        confidence = probabilities[0][predicted.item()].item()
        return "正面" if predicted.item() == 1 else "负面", confidence

# 示例预测
sample_texts = [
    "This movie is absolutely fantastic! The acting was superb and the plot was engaging.",
    "This is the worst movie I have ever seen. The acting was terrible and the plot made no sense.",
    "The movie was okay. Not great, but not terrible either.",
    "I loved this film! The characters were well-developed and the story was compelling.",
    "Boring and predictable. I wouldn't recommend it to anyone."
]

print("\n情感预测示例:")
for i, text in enumerate(sample_texts):
    lstm_prediction, lstm_confidence = predict_sentiment(lstm_model, text, vocab, MAX_SEQUENCE_LENGTH)
    gru_prediction, gru_confidence = predict_sentiment(gru_model, text, vocab, MAX_SEQUENCE_LENGTH)
    print(f"\n示例 {i+1}:")
    print(f"文本: {text}")
    print(f"LSTM预测: {lstm_prediction} (置信度: {lstm_confidence:.4f})")
    print(f"GRU预测: {gru_prediction} (置信度: {gru_confidence:.4f})")

# 性能比较
print(f"\n模型性能比较:")
print(f"LSTM测试准确率: {lstm_accuracy:.2f}%")
print(f"GRU测试准确率: {gru_accuracy:.2f}%")

# 保存词汇表以供后续使用
import pickle
with open('vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)

# 保存最佳模型
torch.save(lstm_model.state_dict(), 'best_lstm_model.pth')
torch.save(gru_model.state_dict(), 'best_gru_model.pth')

# 分析错误预测
def analyze_errors(model, test_loader, vocab, max_length):
    model.eval()
    errors = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs.data, 1)
            
            # 找出错误预测
            incorrect_mask = predicted != labels
            incorrect_indices = torch.where(incorrect_mask)[0]
            
            for idx in incorrect_indices:
                # 获取原始文本（需要从索引重建）
                text_indices = inputs[idx].cpu().numpy()
                # 过滤掉填充标记
                text_indices = text_indices[text_indices != vocab['<pad>']]
                # 将索引转换回单词
                reverse_vocab = {v: k for k, v in vocab.items()}
                words = [reverse_vocab.get(i, '<unk>') for i in text_indices]
                text = ' '.join(words)
                
                errors.append({
                    'text': text,
                    'true_label': 'Positive' if labels[idx].item() == 1 else 'Negative',
                    'predicted_label': 'Positive' if predicted[idx].item() == 1 else 'Negative',
                    'confidence': probabilities[idx][predicted[idx].item()].item()
                })
    
    return errors

# 分析错误
print("\n分析错误预测...")
lstm_errors = analyze_errors(lstm_model, test_loader, vocab, MAX_SEQUENCE_LENGTH)
print(f"LSTM模型错误预测数量: {len(lstm_errors)}")

# 打印一些错误示例
print("\nLSTM模型错误预测示例:")
for i, error in enumerate(lstm_errors[:5]):
    print(f"\n错误 {i+1}:")
    print(f"文本: {error['text'][:100]}...")
    print(f"真实标签: {error['true_label']}")
    print(f"预测标签: {error['predicted_label']}")
    print(f"置信度: {error['confidence']:.4f}")


Training LSTM model...
Epoch [1/15], lstm
Train Loss: 0.6802, Train Acc: 57.02%
Val Loss: 0.5769, Val Acc: 71.33%
Learning Rate: 0.001000
Time: 13.66s
------------------------------------------------------------
Epoch [2/15], lstm
Train Loss: 0.6232, Train Acc: 65.68%
Val Loss: 0.5125, Val Acc: 74.12%
Learning Rate: 0.001000
Time: 13.62s
------------------------------------------------------------
Epoch [3/15], lstm
Train Loss: 0.4563, Train Acc: 81.38%
Val Loss: 0.4163, Val Acc: 82.08%
Learning Rate: 0.001000
Time: 13.74s
------------------------------------------------------------
Epoch [4/15], lstm
Train Loss: 0.3106, Train Acc: 88.67%
Val Loss: 0.3707, Val Acc: 85.17%
Learning Rate: 0.001000
Time: 13.71s
------------------------------------------------------------
Epoch [5/15], lstm
Train Loss: 0.2306, Train Acc: 91.74%
Val Loss: 0.3964, Val Acc: 85.30%
Learning Rate: 0.001000
Time: 13.74s
------------------------------------------------------------
Epoch [6/15], lstm
Train Loss: 0

可视化

In [12]:
# （bash） tensorboard --logdir=runs